In [2]:
import random
import base64
import os
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import padding, hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives.asymmetric import rsa, padding as rsa_padding


In [3]:
# Define a pool of unique characters from various languages
character_pool = [
    # 1. Greek (excluding symbols resembling English letters)
    'Δ', 'Λ', 'Ξ', 'Π', 'Σ', 'Φ', 'Χ', 'Ψ', 'Ω', 'δ', 'λ', 'ξ', 'π', 'σ', 'φ', 'χ', 'ψ', 'ω',
    
    # 2. Russian (excluding symbols resembling English letters)
    'Д', 'Ж', 'З', 'Ц', 'Ч', 'Ш', 'Щ', 'Ъ', 'Ы', 'Ь', 'Э', 'Ю', 'Я', 'д', 'ж', 'з', 'ц', 'ч', 'ш', 'щ', 'ъ', 'ы', 'ь', 'э', 'ю', 'я',
    
    # 3. Japanese Hiragana
    'あ', 'い', 'う', 'え', 'お', 'か', 'き', 'く', 'け', 'こ', 'さ', 'し', 'す', 'せ', 'そ', 'た', 'ち', 'つ', 'て', 'と',
    
    # 4. Arabic (excluding symbols resembling English letters)
    'ث', 'ح', 'خ', 'ذ', 'ز', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ي',
    
    # 5. Georgian
    'ა', 'ბ', 'გ', 'დ', 'ე', 'ვ', 'ზ', 'თ', 'ი', 'კ', 'ლ', 'მ', 'ნ', 'ო', 'პ', 'ჟ', 'რ', 'ს', 'ტ', 'უ', 'ფ', 'ქ', 'ღ', 'ყ', 'შ', 'ჩ', 'ც', 'ძ', 'წ', 'ჭ', 'ხ', 'ჯ', 'ჰ',

    # 6. Tamil
    'அ', 'ஆ', 'இ', 'ஈ', 'உ', 'ஊ', 'எ', 'ஏ', 'ஐ', 'ஒ', 'ஓ', 'ஔ', 'க', 'ங', 'ச', 'ஜ', 'ஞ', 'ட', 'ண', 'த', 'ந', 'ன', 'ப', 'ம', 'ய', 'ர', 'ல', 'ள', 'ழ', 'வ', 'ஷ', 'ஸ', 'ஹ',
    
    # 7. Thai
    'ก', 'ข', 'ฃ', 'ค', 'ฅ', 'ฆ', 'ง', 'จ', 'ฉ', 'ช', 'ซ', 'ฌ', 'ญ', 'ฎ', 'ฏ', 'ฐ', 'ฑ', 'ฒ', 'ณ', 'ด', 'ต', 'ถ', 'ท', 'ธ', 'น', 'บ', 'ป', 'ผ', 'ฝ', 'พ', 'ฟ', 'ภ', 'ม', 'ย', 'ร', 'ฤ', 'ล', 'ฦ', 'ว', 'ศ', 'ษ', 'ส', 'ห', 'ฬ', 'อ', 'ฮ',

    # 8. Armenian
    'Ա', 'Բ', 'Գ', 'Դ', 'Ե', 'Զ', 'Է', 'Ը', 'Թ', 'Ժ', 'Ի', 'Լ', 'Խ', 'Ծ', 'Կ', 'Հ', 'Ձ', 'Ղ', 'Ճ', 'Մ', 'Յ', 'Ն', 'Շ', 'Ո', 'Չ', 'Պ', 'Ջ', 'Ռ', 'Ս', 'Վ', 'Տ', 'Ր', 'Ց', 'Փ', 'Ք', 'Օ', 'Ֆ',
    
    # 9. Hebrew
    'א', 'ב', 'ג', 'ד', 'ה', 'ו', 'ז', 'ח', 'ט', 'י', 'כ', 'ל', 'מ', 'נ', 'ס', 'ע', 'פ', 'צ', 'ק', 'ר', 'ש', 'ת',
    
    # 10. Korean Hangul
    'ㄱ', 'ㄴ', 'ㄷ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅅ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ', 'ㅏ', 'ㅑ', 'ㅓ', 'ㅕ', 'ㅗ', 'ㅛ', 'ㅜ', 'ㅠ', 'ㅡ', 'ㅣ',
    
    # 11. Devanagari (used in Hindi, Sanskrit)
    'अ', 'आ', 'इ', 'ई', 'उ', 'ऊ', 'ऋ', 'ए', 'ऐ', 'ओ', 'औ', 'क', 'ख', 'ग', 'घ', 'ङ', 'च', 'छ', 'ज', 'झ', 'ञ', 'ट', 'ठ', 'ड', 'ढ', 'ण', 'त', 'थ', 'द', 'ध', 'न', 'प', 'फ', 'ब', 'भ', 'म', 'य', 'र', 'ल', 'व', 'श', 'ष', 'स', 'ह',

    # 12. Bengali
    'অ', 'আ', 'ই', 'ঈ', 'উ', 'ঊ', 'ঋ', 'এ', 'ঐ', 'ও', 'ঔ', 'ক', 'খ', 'গ', 'ঘ', 'ঙ', 'চ', 'ছ', 'জ', 'ঝ', 'ঞ', 'ট', 'ঠ', 'ড', 'ঢ', 'ণ', 'ত', 'থ', 'দ', 'ধ', 'ন', 'প', 'ফ', 'ব', 'ভ', 'ম', 'য', 'র', 'ল', 'শ', 'ষ', 'স', 'হ',
    
    # 13. Gujarati
    'અ', 'આ', 'ઇ', 'ઈ', 'ઉ', 'ઊ', 'ઋ', 'એ', 'ઐ', 'ઓ', 'ઔ', 'ક', 'ખ', 'ગ', 'ઘ', 'ચ', 'છ', 'જ', 'ઝ', 'ટ', 'ઠ', 'ડ', 'ઢ', 'ત', 'થ', 'દ', 'ધ', 'ન', 'પ', 'ફ', 'બ', 'ભ', 'મ', 'ય', 'ર', 'લ', 'વ', 'શ', 'ષ', 'સ', 'હ',
    
    # 14. Kannada
    'ಅ', 'ಆ', 'ಇ', 'ಈ', 'ಉ', 'ಊ', 'ಋ', 'ಎ', 'ಏ', 'ಐ', 'ಒ', 'ಓ', 'ಔ', 'ಕ', 'ಖ', 'ಗ', 'ಘ', 'ಚ', 'ಜ', 'ಞ', 'ಟ', 'ಠ', 'ಡ', 'ತ', 'ಥ', 'ದ', 'ನ', 'ಪ', 'ಫ', 'ಬ', 'ಭ', 'ಮ', 'ಯ', 'ರ', 'ಲ', 'ವ', 'ಶ', 'ಸ', 'ಹ',
    
    # 15. Telugu
    'అ', 'ఆ', 'ఇ', 'ఈ', 'ఉ', 'ఊ', 'ఋ', 'ఎ', 'ఏ', 'ఐ', 'ఒ', 'ఓ', 'ఔ', 'క', 'గ', 'ఘ', 'చ', 'జ', 'ఞ', 'ట', 'డ', 'త', 'ద', 'న', 'ప', 'బ', 'మ', 'య', 'ర', 'ల', 'వ', 'శ', 'స', 'హ',

    # 16. Tibetan
    'ཀ', 'ཁ', 'ག', 'ང', 'ཅ', 'ཆ', 'ཇ', 'ཉ', 'ཏ', 'ཐ', 'ད', 'ན', 'པ', 'ཕ', 'བ', 'མ', 'ཙ', 'ཚ', 'ཛ', 'ཝ', 'ཞ', 'ཟ', 'འ', 'ཡ', 'ར', 'ལ', 'ཤ', 'ས', 'ཧ', 'ཨ',

    # 17. Lao
    'ກ', 'ຂ', 'ຄ', 'ງ', 'ຈ', 'ຊ', 'ຍ', 'ດ', 'ຕ', 'ນ', 'ບ', 'ປ', 'ຜ', 'ຝ', 'ພ', 'ຟ', 'ມ', 'ຢ', 'ຣ', 'ລ', 'ວ', 'ສ', 'ຫ', 'ອ',

    # 18. Burmese
    'က', 'ခ', 'ဂ', 'ဃ', 'င', 'စ', 'ဆ', 'ဇ', 'ဈ', 'ဉ', 'ည', 'တ', 'ထ', 'ဒ', 'ဓ', 'န', 'ပ', 'ဖ', 'ဗ', 'ဘ', 'မ', 'ယ', 'ရ', 'လ', 'ဝ', 'သ', 'ဟ', 'ဠ', 'အ',

    # 19. Khmer
    'ក', 'ខ', 'គ', 'ឃ', 'ង', 'ច', 'ឆ', 'ជ', 'ញ', 'ដ', 'ឋ', 'ឌ', 'ឍ', 'ណ', 'ត', 'ថ', 'ទ', 'ធ', 'ន', 'ប', 'ផ', 'ព', 'ភ', 'ម', 'យ', 'រ', 'ល', 'វ', 'ស', 'ហ', 'អ',

    # 20. Sinhala
    'අ', 'ආ', 'ඇ', 'ඈ', 'ඉ', 'ඊ', 'උ', 'ඌ', 'එ', 'ඒ', 'ඓ', 'ඔ', 'ඕ', 'ඖ', 'ක', 'ඛ', 'ග', 'ඝ', 'ච', 'ඡ', 'ජ', 'ඣ', 'ට', 'ඨ', 'ඩ', 'ඪ', 'ත', 'ථ', 'ද', 'ධ', 'න', 'ප', 'ඵ', 'බ', 'භ', 'ම', 'ය', 'ර', 'ල', 'ව', 'ශ', 'ෂ', 'ස', 'හ'
]


# Define an English substitution character pool
english_characters = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"

# Step 1: Multilingual Substitution Layer
def multilingual_substitution(message):
    encrypted_message = ""
    key_map = {}
    
    for char in message:
        if char not in key_map:
            substitute = random.choice(character_pool)
            while substitute in key_map.values():
                substitute = random.choice(character_pool)
            key_map[char] = substitute
        encrypted_message += key_map[char]
    return encrypted_message, key_map

# Step 2: English Substitution Layer
def english_substitution(multilingual_message):
    english_message = ""
    english_map = {}
    
    for char in multilingual_message:
        if char not in english_map:
            substitute = random.choice(english_characters)
            while substitute in english_map.values():
                substitute = random.choice(english_characters)
            english_map[char] = substitute
        english_message += english_map[char]
    return english_message, english_map

# Generate RSA keys (public key used by sender)
def generate_rsa_keys():
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048, backend=default_backend())
    public_key = private_key.public_key()
    return private_key, public_key

# Encrypt the AES key with RSA
def rsa_encrypt_aes_key(aes_key, public_key):
    encrypted_key = public_key.encrypt(
        aes_key,
        rsa_padding.OAEP(mgf=rsa_padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
    )
    return encrypted_key

# Decrypt the AES key with RSA
def rsa_decrypt_aes_key(encrypted_key, private_key):
    aes_key = private_key.decrypt(
        encrypted_key,
        rsa_padding.OAEP(mgf=rsa_padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
    )
    return aes_key

# AES Encryption Layer
def aes_encrypt(data, password, public_key):
    salt = os.urandom(16)
    kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=100000, backend=default_backend())
    aes_key = kdf.derive(password.encode())

    encrypted_aes_key = rsa_encrypt_aes_key(aes_key, public_key)
    iv = os.urandom(16)
    cipher = Cipher(algorithms.AES(aes_key), modes.CBC(iv), backend=default_backend())
    encryptor = cipher.encryptor()

    padder = padding.PKCS7(128).padder()
    padded_data = padder.update(data.encode()) + padder.finalize()

    encrypted_data = encryptor.update(padded_data) + encryptor.finalize()
    return base64.b64encode(salt + iv + encrypted_aes_key + encrypted_data).decode()

# AES Decryption Layer
def aes_decrypt(encrypted_data, password, private_key):
    data = base64.b64decode(encrypted_data)
    salt = data[:16]
    iv = data[16:32]
    encrypted_aes_key = data[32:32 + 256]
    encrypted_message = data[32 + 256:]

    aes_key = rsa_decrypt_aes_key(encrypted_aes_key, private_key)

    cipher = Cipher(algorithms.AES(aes_key), modes.CBC(iv), backend=default_backend())
    decryptor = cipher.decryptor()
    padded_data = decryptor.update(encrypted_message) + decryptor.finalize()

    unpadder = padding.PKCS7(128).unpadder()
    data = unpadder.update(padded_data) + unpadder.finalize()
    return data.decode()

# Reverse English Substitution Layer
def reverse_english_substitution(english_message, english_map):
    reversed_map = {v: k for k, v in english_map.items()}
    multilingual_message = "".join(reversed_map.get(char, char) for char in english_message)
    return multilingual_message

# Reverse Multilingual Substitution Layer
def reverse_multilingual_substitution(multilingual_message, key_map):
    reversed_map = {v: k for k, v in key_map.items()}
    decrypted_message = "".join(reversed_map.get(char, char) for char in multilingual_message)
    return decrypted_message

# Example Usage
message = input("Enter your message:")
password = "Xq7!zF0$N9s@L3#rP5&wT8*UdV1%jK6^mH4(Yb)Q2+GxZ"

# Generate RSA keys
private_key, public_key = generate_rsa_keys()

# Encrypt the message
print("Original message:", message)
multilingual_message, multilingual_map = multilingual_substitution(message)
print("Encrypted message after multilingual substitution:", multilingual_message)
english_message, english_map = english_substitution(multilingual_message)
print("Encrypted message after English substitution:", english_message)
final_encrypted_message = aes_encrypt(english_message, password, public_key)
print("After AES + RSA encryption:", final_encrypted_message)

# Decrypt the message
aes_decrypted_message = aes_decrypt(final_encrypted_message, password, private_key)
multilingual_message = reverse_english_substitution(aes_decrypted_message, english_map)
final_decrypted_message = reverse_multilingual_substitution(multilingual_message, multilingual_map)
print("Final decrypted message:", final_decrypted_message)


Enter your message: 1. **The morning sun cast a warm, amber glow over the quiet cityscape, illuminating the scattered dew on the leaves and rooftops.** Each building, tall and short, held a story, a blend of bustling urban life and serene dawn silence. People began to stir, the hum of traffic gradually picking up as they made their way to work, school, or leisurely routines. The air held a slight chill, a reminder that autumn was creeping in, leaving a crispness that hinted at shorter days and cozy evenings. It's in these early hours that the city felt alive in a different way, still a bit sleepy yet full of potential, ready to embrace whatever the day might bring.  2. **In a world where technology advances at lightning speed, our everyday lives are increasingly intertwined with artificial intelligence.** From personalized recommendations on streaming platforms to sophisticated medical diagnostics, AI is transforming industries in ways once relegated to science fiction. Yet, as powerfu

Original message: 1. **The morning sun cast a warm, amber glow over the quiet cityscape, illuminating the scattered dew on the leaves and rooftops.** Each building, tall and short, held a story, a blend of bustling urban life and serene dawn silence. People began to stir, the hum of traffic gradually picking up as they made their way to work, school, or leisurely routines. The air held a slight chill, a reminder that autumn was creeping in, leaving a crispness that hinted at shorter days and cozy evenings. It's in these early hours that the city felt alive in a different way, still a bit sleepy yet full of potential, ready to embrace whatever the day might bring.  2. **In a world where technology advances at lightning speed, our everyday lives are increasingly intertwined with artificial intelligence.** From personalized recommendations on streaming platforms to sophisticated medical diagnostics, AI is transforming industries in ways once relegated to science fiction. Yet, as powerful 